# Can a policy that reads only the context choose options that beat EI? One-deviation option policy from exact labels

## Description

<!-- Written 2026-11-12 before the first run (pre-registration). Reasoning: research_log/decisions/0008 §4 step 3, and the discussion with Tim of 2026-11-12 after option_headroom (notebook 12). Issue #13. -->

**Question.** Notebook 12 found a luck-corrected advantage over EI in "follow arm a for k steps, then EI" (H₃ = 0.195 log10), about half of it context-dependent. That headroom is privileged: the option was chosen with knowledge of the task. **Can a policy that reads only what is known at decision time (the data and the frozen PFN's belief) choose when to deviate and to which option, and beat EI by more than a context-free schedule does?**

Tim's framing (2026-11-12): the aim is to *discover a non-static acquisition rule that decides from the context*. A better static arm (MES, UCB) beats EI under log regret anyway, and would hide whether the algorithm works. So:
- the base policy stays **EI**;
- **MES is not among the options of the primary policy**, so "switch to MES" cannot be the answer;
- the claim that counts is the gain over the **best context-free schedule** from the same options, not the gain over EI alone.

**Setup** (`experimental/option_policy.py`, `tasks/option_policy.py`, `analysis/option_policy.py`). d = 2, n_init = 4, T = 20, pfn_large, fp32, noise-free prior tasks.
- *Deterministic arms*, as in notebook 12: each arm maximizes its acquisition over the seed-s candidate set (1024 scrambled Sobol points plus 64 fixed perturbations of the incumbent at each of σ = 0.01, 0.05, 0.2). Seeds s ∈ {0, 1} are the luck replicates.
- *Tasks.* 2000 fresh prior draws: split 4, task offset 30000 (label tasks; notebook 12 used 20000–20799). Nothing touches split 1.
- *Prefix.* EI under seed 0 for the whole run.
- *Decision steps.* t ∈ {0, 2, …, 14}. At each, from the prefix's data, under both seeds:
  - the **base**: EI to T;
  - the **options**: each of PI(0.01), UCB(0.995), MES, mean for 3 steps and then EI to T, or for the rest of the run. 8 options.
- *Reward.* Terminal log10 regret, log10(best − f\* + ε), ε = 1e-6. The advantage of an option under seed s is log10 R_T(base, s) − log10 R_T(option, s); positive = better. Each is an exact outcome (deterministic policies, noise-free tasks).
  - Under seed 0 the base from every step is the prefix itself, so the advantage is the exact gain over the EI run.
- *Context*, computed at every (task, t) from the prefix's data only:
  - `data` (18 features): budget, the y values' level and gaps, the improvement history, the geometry around the incumbent;
  - `belief` (data + 45): the PFN's predictive on the seed-0 candidate set (max EI, max PI, uncertainty near and far, MES's f\* samples), and for the pick of each of EI, PI, UCB, MES, mean: its distance to the incumbent and to EI's pick, its predicted mean, sd, PI and EI;
  - `pfn` (belief + 32): the first 32 principal components of the PFN's pooled last-layer context states;
  - `time` (t only): a learned schedule that cannot see the context.
- *Value model.* Q(state, option): an MLP (2 × 128, ensemble of 2) regressed by squared error on the advantage averaged over both seeds. A ridge on `belief` as the linear reference.
- *Policy (one deviation).* Follow EI. At the first decision step where max over the portfolio's options of Q exceeds c, commit to that option; afterwards EI to T, no second deviation.
  - Every state the policy decides at is an EI state, so the labels are its exact value: nothing compounds and no off-policy correction is needed.
  - **Conservative acceptance:** c is chosen to maximize the value on training tasks, from predictions those tasks' models never saw. c = ∞ (never deviate = EI) is in the grid and wins ties.
- *Cross-fitting over tasks.* 5 folds. Each task's decision comes from a model and a threshold fitted on the other four folds (thresholds from 4 inner folds inside them). The policy's value on a task is the label of its decision. All 2000 tasks are held-out test tasks once.
- *Portfolios.* `nomes` (**primary**): PI, UCB, mean. `all`: plus MES.
- *Comparators*, all on the same held-out tasks:
  - **EI**: 0 by definition;
  - **static schedule**: the single best (step, option) on the training folds, applied to every held-out task. "Always UCB from step 0" is one of them;
  - **shuffled null**: the policy's own decisions reassigned across held-out tasks. Same deviation rate, steps and options, unrelated to the task's context;
  - **`time` policy**: the same learner without context;
  - **oracle** (privileged): per task the best decision on one seed, scored on the other. The headroom of the one-deviation class.

**What to look for.**
1. **Primary:** `nomes/belief/mlp` under seed 0 (the deployed, deterministic policy; exact closed-loop gain):
   - gain over EI, paired over tasks, bootstrap CI;
   - **gain over the static schedule** = the context-dependent part;
   - gain over its shuffled null.
2. Secondary:
   - the same under seed 1 (option executed on a different candidate set: a luck replicate of the decision);
   - `all` portfolio: does the policy use MES, and does the context still add to it?
   - feature sets: `time` < `data` < `belief` < `pfn`? `belief` vs `data` is the PFN-reading vs D-only comparison;
   - share of the oracle headroom captured;
   - which steps and options the policy chooses, and its deviation rate;
   - held-out fit of Q per option against the label's cross-seed correlation (its noise ceiling);
   - robustness at ε = 1e-5, from the stored raw regrets.

**GO / NO-GO** (rule `analysis/option_policy.py::policy_verdict`, on `nomes/belief/mlp`, seed 0; worthwhile = 0.022 log10, i.e. 5% less regret).
- **GO**: the gain over EI has CI > 0, and the gain over the static schedule has CI > 0 and is ≥ 0.022. A context-reading policy beats EI beyond what a context-free schedule gives. Next: confirm with a frozen policy in closed loop on split 1; allow repeated decisions; d = 4.
- **NO-GO**: the upper CI bound of the gain over the static schedule is < 0.022. The context, as read here, does not identify the option. Next: d = 4, where the headroom should be basin discovery; or richer state readouts.
- **INCONCLUSIVE**: otherwise.

A gain over EI without a gain over the static schedule is *not* a GO: it is "a better default rule than EI", which notebook 12 already showed.

**Limits.**
- Noise-free P0, d = 2, T = 20; at d = 2 the gain is mostly polishing efficiency (notebook 12).
- One deviation per run, at even steps only. Repeated decisions would need labels at the states the new policy visits (the next iteration).
- Q compares "deviate now" with "never deviate", not with "deviate later"; the threshold absorbs part of that.
- The value is exact for seed 0, but offline: read from labels, not from a separate closed-loop run. The confirm stage runs the frozen policy in closed loop.
- Pilot on label tasks (split 4), cross-fitted. Design choices (features, learner, c grid) were fixed before the first run; the 32 PCA directions are fitted on all states' PFN states (no labels involved).
- Two seeds: training targets average them, a coarse luck reduction.


### Note after the sanity run (2026-11-12, before the pilot)

Sanity: `results/option_policy/sanity_20261112-123006`, 100 tasks, 6 min. The pipeline works. No change to the design or the rule.

One addition to what is reported: under seed 0 the base lane from every decision step should reproduce the prefix exactly. It does so only up to GPU floating-point effects: raw regrets differ by at most 7e-6, which is more than 0.01 log10 in 27% of tasks, since near-ties late in a run flip with the batch composition. The labels are therefore exact up to this mean-zero numerical noise, at the scale of ε. The spread of the base lanes across steps is reported as that noise scale.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from anytime_acq.analysis import load_snapshot, show_header
from anytime_acq.analysis.option_policy import policy_verdict
from anytime_acq.analysis.stats import bootstrap_ci

SNAPSHOT = "results/option_policy/sanity_20261112-123006"
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

In [ ]:
runs = snap.runs
print("member status:", runs.status.value_counts().to_dict())
print("minutes:", {k: round(float(runs.iloc[0][k]) / 60, 1) for k in runs.columns if k.startswith("time/")})
summ, per, usage, fit, opts = (snap.table(k) for k in ("summary", "per_task", "usage", "fit", "options"))
PRIMARY, WORTH = "nomes/belief/mlp", 0.022
print("held-out tasks:", len(per), "| folds:", per.fold.nunique())

## 1. The primary policy: gain over EI, over the static schedule, over its shuffled null

In [ ]:
def show(variants, seed=0):
    s = summ[summ.variant.isin(variants) & (summ.seed == seed)]
    t = s.assign(ci=[f"{m:+.3f} [{lo:+.3f}, {hi:+.3f}]" for m, lo, hi in zip(s["mean"], s.lo, s.hi)])
    out = t.pivot(index="variant", columns="contrast", values="ci").reindex(variants)
    out["deviates"] = s.groupby("variant").deviates.first().reindex(variants).round(2)
    return out[[c for c in ("vs_base", "vs_static", "vs_shuffled", "deviates") if c in out]]

print("log10 terminal regret, positive = better; paired over held-out tasks, 95% bootstrap CI")
print("seed 0 (the deployed policy; exact gain over the EI run):")
display(show([PRIMARY, "nomes/static"], 0))
print("seed 1 (the decision executed on a different candidate set):")
display(show([PRIMARY, "nomes/static"], 1))
o = summ[summ.variant == "nomes/oracle"].iloc[0]
g = summ[(summ.variant == PRIMARY) & (summ.seed == 0) & (summ.contrast == "vs_base")].iloc[0]
print(f"privileged one-deviation headroom (cross-fitted over seeds): {o['mean']:.3f} [{o.lo:.3f}, {o.hi:.3f}];"
      f" share captured by the policy: {g['mean'] / o['mean']:.0%}")

## 2. Feature sets, learners and portfolios

In [ ]:
order = [f"{pf}/{fs}/mlp" for pf in ("nomes", "all") for fs in ("time", "data", "belief", "pfn")]
order = order[:4] + ["nomes/belief/ridge", "nomes/static"] + order[4:] + ["all/belief/ridge", "all/static"]
display(show(order, 0))
fig, ax = plt.subplots(1, 2, figsize=(13, 3.6), dpi=90, sharey=True)
for a, contrast in zip(ax, ("vs_base", "vs_static")):
    s = summ[(summ.seed == 0) & (summ.contrast == contrast) & summ.variant.isin(order)].set_index("variant").reindex(order).dropna(subset=["mean"])
    a.errorbar(range(len(s)), s["mean"], yerr=[s["mean"] - s.lo, s.hi - s["mean"]], fmt="o", capsize=3)
    a.axhline(0, color="k", lw=.8); a.axhline(WORTH, color="gray", ls=":", lw=.8)
    a.set_xticks(range(len(s))); a.set_xticklabels(s.index, rotation=60, ha="right", fontsize=8)
    a.set_title({"vs_base": "gain over EI", "vs_static": "gain over the static schedule (context part)"}[contrast]); a.grid(alpha=.3)
ax[0].set_ylabel("log10 regret, positive = better"); plt.show()

## 3. What the policy does: steps, options, thresholds

In [ ]:
u = usage[usage.variant.isin([PRIMARY, "all/belief/mlp"])]
for v, g in u.groupby("variant"):
    print(v, "(tasks per decision; step -1 = never deviates)")
    display(g.pivot_table(index="option", columns="step", values="n", aggfunc="sum", fill_value=0))
print("static schedule chosen per fold:"); display(usage[usage.variant.str.endswith("/static")][["variant", "fold", "step", "option"]])
print("thresholds c per fold (primary):", sorted(per.groupby("fold")[f"{PRIMARY}:c"].first().round(3).tolist()))
print("mean advantage per decision step and option (seed 0):")
display(opts.pivot(index="option", columns="step", values="adv0").round(3))

## 4. How well Q is fitted, against the labels' noise ceiling

In [ ]:
f = fit.pivot_table(index="option", columns=["features", "learner"], values="spearman").round(3)
f["label seed corr"] = fit.groupby("option").seed_corr.first().round(3)
print("held-out Spearman of Q with the seed-mean advantage, per option; last column: correlation of the label across the two seeds")
display(f)
display(fit.groupby(["features", "learner"])[["spearman", "r2"]].mean().round(3))

## 5. Robustness: ε = 1e-5, and the numerical noise of the labels

In [ ]:
def gain_eps(v, eps, s=0):
    return np.log10(per[f"{v}:regret_base{s}"] + eps) - np.log10(per[f"{v}:regret{s}"] + eps)

rows = []
for v in (PRIMARY, "all/belief/mlp", "nomes/pfn/mlp", "nomes/data/mlp"):
    for eps in (1e-6, 1e-5):
        x = gain_eps(v, eps).to_numpy()
        lo, hi = bootstrap_ci(x, n_boot=2000)
        rows.append({"variant": v, "eps": eps, "gain over EI": x.mean(), "lo": lo, "hi": hi})
print("the same decisions, scored at another eps (the policy itself was trained at eps = 1e-6):")
display(pd.DataFrame(rows).round(3))

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.

In [ ]:
print("pre-registered rule on", PRIMARY, "seed 0:", policy_verdict(summ, PRIMARY, WORTH))